Import Pandas, SQLite, and Path.

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd

Set the raw-data file paths, then preview the results and competitions files.

In [ ]:
raw_folder = Path("../data/raw/WCA_export_v2_258_20260915T000024Z.tsv")
results_path = raw_folder / "WCA_export_results.tsv"
competitions_path = raw_folder / "WCA_export_competitions.tsv"
results_preview = pd.read_csv(results_path, sep="\t", nrows=5)
competitions_preview = pd.read_csv(competitions_path, sep="\t", nrows=5)
display(results_preview)
display(competitions_preview)

Review the available columns before selecting the fields needed for the analysis.

In [ ]:
print("Results rows and columns:", results_preview.shape)
print("Results columns:", results_preview.columns.tolist())
print("Competitions rows and columns:", competitions_preview.shape)
print("Competitions columns:", competitions_preview.columns.tolist())

Load the large results file in chunks, keeping valid 3x3 averages and only the needed columns.

In [ ]:
results_columns = ["competition_id", "event_id", "person_name", "person_id", "person_country_id", "best", "average"]
filtered_chunks = []
for chunk in pd.read_csv(results_path, sep="\t", usecols=results_columns, chunksize=500_000):
    valid_3x3_results = chunk[(chunk["event_id"] == "333") & (chunk["average"] > 0)]
    filtered_chunks.append(valid_3x3_results)
results_3x3 = pd.concat(filtered_chunks, ignore_index=True)
print("Valid 3x3 rows:", len(results_3x3))
display(results_3x3.head())

Load competition names and dates, then keep the fields needed for the merge.

In [ ]:
competitions = pd.read_csv(competitions_path, sep="\t", usecols=["id", "name", "year", "month", "day"])
competitions["competition_date"] = pd.to_datetime(competitions[["year", "month", "day"]])
competitions = competitions.rename(columns={"id": "competition_id", "name": "competition_name"})
competitions = competitions[["competition_id", "competition_name", "competition_date"]]
print("Competitions:", len(competitions))
display(competitions.head())

Add the competition name and date to each valid 3x3 result.

In [ ]:
results_with_dates = results_3x3.merge(competitions, on="competition_id", how="left", validate="many_to_one")
print("Rows after merge:", len(results_with_dates))
print("Missing competition dates:", results_with_dates["competition_date"].isna().sum())
display(results_with_dates.head())

Keep each competitor's fastest valid average at each competition and convert WCA centiseconds to seconds.

In [ ]:
results_with_dates = results_with_dates.sort_values("average")
competition_results = results_with_dates.drop_duplicates(subset=["person_id", "competition_id"], keep="first").copy()
competition_results["average_seconds"] = competition_results["average"] / 100
competition_results["best_single_seconds"] = competition_results["best"] / 100
competition_results = competition_results.sort_values(["person_id", "competition_date", "competition_id"]).reset_index(drop=True)
print("Round-level rows:", len(results_with_dates))
print("Competitor-competition rows:", len(competition_results))
display(competition_results.head())

Number each competitor's competitions and calculate their total competition count.

In [ ]:
competition_results["competition_number"] = competition_results.groupby("person_id").cumcount() + 1
competition_results["total_competitions"] = competition_results.groupby("person_id")["competition_id"].transform("size")
print("Unique competitors:", competition_results["person_id"].nunique())
display(competition_results[["person_id", "competition_number", "total_competitions"]].head())

Keep competitors with at least five valid competition averages, which is the project's analysis sample.

In [ ]:
analysis_results = competition_results[competition_results["total_competitions"] >= 5].copy()
duplicate_count = analysis_results.duplicated(subset=["person_id", "competition_id"]).sum()
print("Competitors:", analysis_results["person_id"].nunique())
print("Records:", len(analysis_results))
print("Duplicate competitor-competition rows:", duplicate_count)
print("First date:", analysis_results["competition_date"].min())
print("Latest date:", analysis_results["competition_date"].max())

Create one summary row per competitor with their first result, best result, and improvement.

In [ ]:
competitor_summary = analysis_results.groupby("person_id", sort=False).agg(person_name=("person_name", "last"), country=("person_country_id", "last"), total_competitions=("total_competitions", "first"), first_competition_date=("competition_date", "first"), last_competition_date=("competition_date", "last"), first_average_seconds=("average_seconds", "first"), best_average_seconds=("average_seconds", "min")).reset_index()
competitor_summary["improvement_seconds"] = competitor_summary["first_average_seconds"] - competitor_summary["best_average_seconds"]
competitor_summary["improvement_percent"] = (competitor_summary["improvement_seconds"] / competitor_summary["first_average_seconds"] * 100).round(1)
competitor_summary["years_competing"] = ((competitor_summary["last_competition_date"] - competitor_summary["first_competition_date"]).dt.days / 365.25).round(1)
print("Summary rows:", len(competitor_summary))
display(competitor_summary.head())

Record whether each competitor reached the sub-30, sub-20, sub-15, and sub-10 milestones.

In [ ]:
milestones = [30, 20, 15, 10]
for milestone in milestones:
    competitions_to_milestone = analysis_results[analysis_results["average_seconds"] < milestone].groupby("person_id")["competition_number"].min()
    competitions_to_milestone = competitions_to_milestone.rename(f"competitions_to_sub_{milestone}")
    competitor_summary = competitor_summary.merge(competitions_to_milestone, on="person_id", how="left", validate="one_to_one")
    competitor_summary[f"reached_sub_{milestone}"] = competitor_summary[f"competitions_to_sub_{milestone}"].notna()
display(competitor_summary.head())

Calculate personal-best progression and summarize the first 10 competitions for the Tableau trend chart.

In [ ]:
analysis_results["personal_best_to_date"] = analysis_results.groupby("person_id")["average_seconds"].cummin()
first_10_results = analysis_results[(analysis_results["total_competitions"] >= 10) & (analysis_results["competition_number"] <= 10)].copy()
path_summary = first_10_results.groupby("competition_number").agg(competitors=("person_id", "nunique"), median_average_seconds=("average_seconds", "median"), median_personal_best_seconds=("personal_best_to_date", "median")).reset_index()
path_summary[["median_average_seconds", "median_personal_best_seconds"]] = path_summary[["median_average_seconds", "median_personal_best_seconds"]].round(2)
display(path_summary)

Summarize milestone achievement rates for the Tableau milestone chart.

In [ ]:
milestone_summary = pd.DataFrame({"milestone": ["Sub-30", "Sub-20", "Sub-15", "Sub-10"], "competitors_reached": [competitor_summary["reached_sub_30"].sum(), competitor_summary["reached_sub_20"].sum(), competitor_summary["reached_sub_15"].sum(), competitor_summary["reached_sub_10"].sum()], "percent_reached": [competitor_summary["reached_sub_30"].mean() * 100, competitor_summary["reached_sub_20"].mean() * 100, competitor_summary["reached_sub_15"].mean() * 100, competitor_summary["reached_sub_10"].mean() * 100], "median_competitions_to_reach": [competitor_summary["competitions_to_sub_30"].median(), competitor_summary["competitions_to_sub_20"].median(), competitor_summary["competitions_to_sub_15"].median(), competitor_summary["competitions_to_sub_10"].median()]})
milestone_summary[["percent_reached", "median_competitions_to_reach"]] = milestone_summary[["percent_reached", "median_competitions_to_reach"]].round(1)
display(milestone_summary)

Run the main quality checks before exporting the final datasets.

In [ ]:
print("Best average is not slower than first average:", (competitor_summary["best_average_seconds"] <= competitor_summary["first_average_seconds"]).all())
print("Competition numbers match total competitions:", (analysis_results.groupby("person_id")["competition_number"].max() == analysis_results.groupby("person_id")["total_competitions"].first()).all())
print("No missing competitor IDs:", analysis_results["person_id"].notna().all())
print("No missing competition dates:", analysis_results["competition_date"].notna().all())
print("All averages are positive:", (analysis_results["average_seconds"] > 0).all())
print("No duplicate competitor-competition rows:", duplicate_count == 0)

Create the final competitor-progression table used for detailed analysis.

In [ ]:
analysis_results["first_competition_date"] = analysis_results.groupby("person_id")["competition_date"].transform("min")
analysis_results["days_since_first_competition"] = (analysis_results["competition_date"] - analysis_results["first_competition_date"]).dt.days
competitor_progression = analysis_results[["person_id", "person_name", "person_country_id", "competition_id", "competition_name", "competition_date", "competition_number", "total_competitions", "average_seconds", "best_single_seconds", "personal_best_to_date", "days_since_first_competition"]].copy()
competitor_progression = competitor_progression.rename(columns={"person_country_id": "country"})
print("Progression table:", competitor_progression.shape)
print("Competitor summary:", competitor_summary.shape)
print("Path summary:", path_summary.shape)
print("Milestone summary:", milestone_summary.shape)
display(competitor_progression.head())

Export the four processed CSV files used by SQL and Tableau.

In [ ]:
processed_folder = Path("../data/processed")
processed_folder.mkdir(parents=True, exist_ok=True)
competitor_progression.to_csv(processed_folder / "competitor_progression.csv", index=False)
competitor_summary.to_csv(processed_folder / "competitor_summary.csv", index=False)
path_summary.to_csv(processed_folder / "path_summary.csv", index=False)
milestone_summary.to_csv(processed_folder / "milestone_summary.csv", index=False)
print("Exported competitor_progression.csv:", len(competitor_progression), "rows")
print("Exported competitor_summary.csv:", len(competitor_summary), "rows")
print("Exported path_summary.csv:", len(path_summary), "rows")
print("Exported milestone_summary.csv:", len(milestone_summary), "rows")

Load the processed tables into SQLite so the SQL queries use the same data as Tableau.

In [ ]:
database_path = Path("../sql/speedcubing_analysis.db")
connection = sqlite3.connect(database_path)
competitor_progression.to_sql("competitor_progression", connection, if_exists="replace", index=False)
competitor_summary.to_sql("competitor_summary", connection, if_exists="replace", index=False)
path_summary.to_sql("path_summary", connection, if_exists="replace", index=False)
milestone_summary.to_sql("milestone_summary", connection, if_exists="replace", index=False)
connection.close()
print("Created SQLite database:", database_path)